# Load and process S. aureus data

In [1]:
import numpy as np
import pandas as pd
from Bio import SeqIO
import re

In [2]:
gene_info_df = pd.read_csv("../data/staph/core_gene_info.csv")
gene_info_df['Gene_Length_MRSA252'] = None
gene_info_df['Start_pos_MRSA252'] = None
gene_info_df['End_pos_MRSA252'] = None
gene_info_df.head()

,Gene_ID,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_MRSA252,Start_pos_MRSA252,End_pos_MRSA252
0,dnaA,1362,517,1878,True,None,None,None
1,dnaN,1134,2156,3289,True,None,None,None
2,SAR0003,246,3670,3915,True,None,None,None
3,recF,1113,3912,5024,True,None,None,None
4,gyrB,1932,5037,6968,True,None,None,None


In [3]:
def parse_location_string(loc_str):
    if not loc_str:
        return None, None, False

    is_complement = "complement" in loc_str

    numbers = re.findall(r'\d+', loc_str)
    start = int(numbers[0]) if len(numbers) >= 1 else None
    end = int(numbers[1]) if len(numbers) >= 2 else None
    
    return start, end, is_complement

In [4]:
MRSA252_path = "../data/staph/MRSA252.txt"
MRSA252_genes = []

for record in SeqIO.parse(MRSA252_path, "fasta"):
    desc = record.description

    gene_match     = re.search(r'\[gene=([^\]]+)\]', desc)
    locus_match    = re.search(r'\[locus_tag=([^\]]+)\]', desc)
    location_match = re.search(r'\[location=([^\]]+)\]', desc)
    
    gene_val     = gene_match.group(1) if gene_match else None
    locus_val    = locus_match.group(1) if locus_match else None
    location_val = location_match.group(1) if location_match else None
    
    start, end, is_complement = parse_location_string(location_val)

    final_name = gene_val if gene_val else locus_val
    if not final_name:
        final_name = record.id
        
    MRSA252_genes.append({
        "name": final_name,
        "locus_tag": locus_val,
        "start": start,
        "end": end,
        "is_complement": is_complement
    })

In [5]:
MRSA252_genes[:5]

[{'name': 'dnaA',
  'locus_tag': 'SAR0001',
  'start': 517,
  'end': 1878,
  'is_complement': False},
 {'name': 'dnaN',
  'locus_tag': 'SAR0002',
  'start': 2156,
  'end': 3289,
  'is_complement': False},
 {'name': 'SAR0003',
  'locus_tag': 'SAR0003',
  'start': 3670,
  'end': 3915,
  'is_complement': False},
 {'name': 'recF',
  'locus_tag': 'SAR0004',
  'start': 3912,
  'end': 5024,
  'is_complement': False},
 {'name': 'gyrB',
  'locus_tag': 'SAR0005',
  'start': 5037,
  'end': 6968,
  'is_complement': False}]

In [6]:
skip_indices = []
for i in range(gene_info_df.shape[0]):
    gene_id = gene_info_df.at[i, 'Gene_ID']
    matching_genes = [gene for gene in MRSA252_genes if gene['name'] == gene_id or gene['locus_tag'] == gene_id]

    if matching_genes and len(matching_genes) == 1:
        gene_info = matching_genes[0]
        gene_info_df.at[i, 'Gene_Length_MRSA252'] = gene_info['end'] - gene_info['start'] + 1
        gene_info_df.at[i, 'Start_pos_MRSA252'] = gene_info['start']
        gene_info_df.at[i, 'End_pos_MRSA252'] = gene_info['end']
    elif matching_genes and len(matching_genes) > 1:
        print(f"Multiple matches found for Gene_ID: {gene_id}.")
        skip_indices.append(i)
    else:
        print(f"No match found for Gene_ID: {gene_id}")
        skip_indices.append(i)

No match found for Gene_ID: kdpE2


In [7]:
skip_indices

[1460]

In [8]:
gene_info_df.iloc[skip_indices]

,Gene_ID,Gene_Length,Start_pos,End_pos,Alignment,Gene_Length_MRSA252,Start_pos_MRSA252,End_pos_MRSA252
1460,kdpE2,696,2237966,2238661,True,None,None,None


In [9]:
gene_info_df = gene_info_df.drop(skip_indices, axis=0)
gene_info_df.shape

(1982, 8)

In [10]:
print(gene_info_df.head())
gene_info_df.to_csv("../data/staph/core_gene_info_updated.csv", index=False)

   Gene_ID  Gene_Length  Start_pos  End_pos  Alignment Gene_Length_MRSA252  \
0     dnaA         1362        517     1878       True                1362   
1     dnaN         1134       2156     3289       True                1134   
2  SAR0003          246       3670     3915       True                 246   
3     recF         1113       3912     5024       True                1113   
4     gyrB         1932       5037     6968       True                1932   

  Start_pos_MRSA252 End_pos_MRSA252  
0               517            1878  
1              2156            3289  
2              3670            3915  
3              3912            5024  
4              5037            6968  


In [11]:
x_obs_df = pd.read_csv("../data/staph/core_gene_summary_stats.csv", index_col=0)
print(x_obs_df.head())
print('kdpE2' in x_obs_df.index)
print(x_obs_df.shape)

                 0         1            2           3           4          5  \
dnaA      7.260992  1.742397   460.045044  164.503327   36.711161   9.595627   
dnaN      5.145702  0.824876   775.783980  120.000000   48.881814   7.015775   
SAR0003   0.004463  0.000000    10.000000    0.000000    0.004865   0.000000   
recF      1.534959  0.203636   360.239170  101.194139   14.712357   1.389081   
gyrB     13.946281  2.480000  1676.565363  365.351926  102.650065  10.595015   

            6     7         8         9  ...         36        37        38  \
dnaA      9.0   3.0  0.015615  0.010560  ...  10.220009 -0.503266  0.028571   
dnaN     12.0   0.0  0.006572  0.006874  ...   6.967684 -1.285856  0.119048   
SAR0003   0.0   0.0  0.000446  0.000000  ...   0.201555 -1.560774  0.000000   
recF      5.0   2.0  0.004229  0.001996  ...   2.980656 -1.694897  0.033333   
gyrB     45.0  19.0  0.008199  0.006544  ...  16.084094 -1.115822  0.078125   

               39   40        41        42  

In [12]:
x_obs_df = x_obs_df.drop('kdpE2', axis=0)
print('kdpE2' in x_obs_df.index)
print(x_obs_df.shape)

False
(1982, 46)


In [14]:
x_obs_df.to_csv("../data/staph/core_gene_summary_stats_updated.csv", index=True)